# Lab 1 · Your first memory, and where it lives

**About 15 minutes.** You'll connect an agent to Oracle AI Agent Memory, give it something to remember, and then look at that memory three ways:
- as rows in a table
- as a search result
- in the memory inspector's dashboard

By the end you'll have run the loop the other labs use: **check → fix → check again**.

**Before you start:** the database container is up, and `agent/scripts/apply_sql.py` has run once (see the repo README). The dashboard is optional but worth it: `cd web && npm run dev`, then open http://localhost:3000.

In [ ]:
import labkit
from oracleagentmemory.core.dbschemapolicy import SchemaPolicy
from oracleagentmemory.core.embedders.embedder import Embedder
from oracleagentmemory.core.llms.llm import Llm
from oracleagentmemory.core.oracleagentmemory import OracleAgentMemory
from memory_inspector import inspect

USER = labkit.lab_user(1)
print("database user:", labkit.DB_USER, "· memory user:", USER, "· model:", labkit.LLM_MODEL)

## 1. Build the client, and wrap it

The memory client needs three things: a connection, an embedder and an LLM. The LLM is used for **extraction**, turning conversation into durable memories.

Two details matter here:
- **`schema_policy=CREATE_IF_NECESSARY`.** The default only validates an existing schema, and fails on a fresh one.
- **`inspect(...)`** is the memory inspector. It's one line, it changes nothing about how the client behaves, and from here on every turn is recorded: what was searched, what came back, and what memory changed.

In [ ]:
pool = labkit.open_pool()
client = OracleAgentMemory(
    connection=pool,
    embedder=Embedder(model=labkit.EMBED_MODEL),
    llm=Llm(model=labkit.LLM_MODEL),
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
)
memory = inspect(client, pool=pool)   # the one line

labkit.reset_user(memory, USER)        # so this lab always starts empty

## 2. Have a conversation

A **thread** is one conversation. `add_messages` stores the exchange and runs extraction **inline**, so it takes a few seconds: that's an LLM deciding what is worth keeping.

In [ ]:
thread = memory.create_thread(user_id=USER, agent_id="lab_agent")
message_ids = thread.add_messages([
    {"role": "user", "content": "I'm Sam. I run the data platform team, and I'd rather get status updates in Slack than by email."},
    {"role": "assistant", "content": "Got it, Sam: status updates go to Slack."},
])

You can also write a memory directly. With no `thread_id` it belongs to the user, not to any one conversation. Keep that difference in mind for lab 2.

In [ ]:
memory_id = memory.add_memory("Sam's team owns the nightly export pipeline.", user_id=USER, memory_type="fact")

## 3. Memory is rows in your database

There's no hidden service. Here is what extraction kept, read straight from the `MEMORY` table:

In [ ]:
labkit.show_memories(pool, USER)

with pool.acquire() as conn:
    for memory_type, thread_id, content in conn.cursor().execute(
        "select memory_type, thread_id, dbms_lob.substr(content, 60, 1) from memory where user_id = :u", u=USER
    ):
        print(f"{memory_type:<10} thread_id={thread_id or 'NULL (user level)':<40} {content}")

Look at `thread_id`. The memories extraction wrote are stored **on the thread**. The one you wrote yourself is stored at **user level**. That decides what survives when a conversation is deleted, and lab 2 shows why it matters.

## 4. Search, the way an agent retrieves

An agent asks memory a question before it replies. Each result comes back with a **cosine distance**: lower is closer, and 0 means identical in meaning. The package doesn't store these distances anywhere. The inspector records them for you.

In [ ]:
question = "How should I send Sam the weekly status?"
results = memory.search(question, user_id=USER, record_types=labkit.MEMORY_TYPES, max_results=5)
labkit.show_results(results, highlight="slack")

Now finish the turn the way an agent does, by writing the exchange. The search and the write become one recorded turn.

In [ ]:
message_ids = thread.add_messages([
    {"role": "user", "content": question},
    {"role": "assistant", "content": "Post it in Slack. Sam prefers Slack to email."},
])
print("this conversation:     ", labkit.run_url(thread.thread_id))
print("why the reply said that:", labkit.run_url(thread.thread_id, turn=2, view="why"))

Open the **why** link. It shows every search result for that turn, its distance, and which turn created it. That's the question you'll ask of every surprising reply from now on.

## 5. Check the memory's health

`check` looks for stale, duplicate and transient memories. It uses `VECTOR_DISTANCE` inside the database, plus an LLM judge for the close calls. A store this small should come back clean, or nearly. Labs 2 and 3 will not.

In [ ]:
report = labkit.check(pool, USER)

In [ ]:
memory.close()
pool.close()

## What you saw

- Memory is **rows in Oracle**: `MEMORY` for durable memories, `MESSAGE` for the conversation, and `RECORD_CHUNKS` for their embeddings. You can query it, grant access to it, and delete it like any other data.
- Extraction decides what becomes durable. You didn't choose those memories; an LLM did.
- `inspect()` records each turn without changing the agent, and the dashboard's **why** view answers "why did it say that?"
- `check` is the other half: a health report across the whole store.

**Next: lab 2**, where extraction keeps things it shouldn't, and you fix it.